# LeAgents quickstart — one autonomous cycle on a free Colab GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ratelcode/LeAgents/blob/main/notebooks/quickstart_colab.ipynb)

This notebook runs one complete **collect → train → eval → decide** cycle of
[LeAgents](https://github.com/ratelcode/LeAgents) on
[PushT](https://huggingface.co/datasets/lerobot/pusht) — a 2D task that needs no
MuJoCo/EGL setup. The policy is deliberately under-trained (30 steps): the point is
to watch the *loop* work — the constitution safety gate, the real
`lerobot-train`/`lerobot-eval` subprocesses, the decision function, the event log,
and the knowledge pages. ~5 minutes total on a T4.

> **Runtime → Change runtime type → T4 GPU.** (The dry-run cell also works on CPU.)


In [ ]:
%pip install -q leagents "lerobot[pusht]>=0.5,<0.6"  # ~3 min


## Configure the loop

A run is fully declarative: a YAML config plus a *constitution* — the safety gate
every agent command must pass (sim-only envs, step/episode caps, forbidden
real-hardware entrypoints). Verdicts are written to the event log for audit.


In [ ]:
from pathlib import Path

Path("constitution.yaml").write_text('''# Constitution safety gate — sim-only (DESIGN.md section 3.1 and 6).
allowed_env_types: [libero, metaworld, pusht]
max_train_steps: 200000
max_eval_episodes: 500

forbidden_commands:
  - lerobot-record          # real-hardware collection (M3)
  - lerobot-teleoperate
  - lerobot-async-inference # CVE-2026-25874: pickle RCE in gRPC path

forbidden_args:
  - --robot                 # real-hardware drivers (M3)
  - --teleop
''')
Path("smoke_pusht.yaml").write_text('''# One short cycle on PushT (2D, no MuJoCo/EGL) — validates the loop, not skill.
run_name: smoke-pusht
workdir: runs
seed_dataset: lerobot/pusht

policy_ladder:
  - name: diffusion   # from scratch; canonical pairing with lerobot/pusht

budgets:
  max_cycles: 1
  max_gpu_hours: 1

thresholds:
  promote_delta: 0.05
  regression_delta: 0.05
  plateau_epsilon: 0.01
  plateau_cycles: 2

train:
  steps: 30
  batch_size: 8
  extra_args: ["--policy.device=cuda", "--num_workers=2", "--log_freq=10", "--save_freq=30"]

eval:
  env_type: pusht
  task_suite: PushT-v0
  n_episodes: 2
  extra_args: ["--policy.device=cuda"]

llm: null
knowledge:
  enabled: true
  root: knowledge

constitution: constitution.yaml
''')
print("configs written")


## Dry run — no GPU needed

Synthetic eval scores exercise the whole state machine (budgets, decision
function, SQLite store, events) without touching lerobot:


In [ ]:
!leagents run -c smoke_pusht.yaml --dry-run


## Real cycle — train + eval on the GPU (~2 min)

Trains a diffusion policy for 30 steps on `lerobot/pusht`, evaluates 2 episodes
with `lerobot-eval`, then the deterministic decision function rules
promote / iterate / escalate / rollback:


In [ ]:
!leagents run -c smoke_pusht.yaml


In [ ]:
!leagents status


## What happened — decisions, events, and the eval video


In [ ]:
import json
from pathlib import Path

run = sorted(Path("runs").glob("*smoke-pusht"))[-1]
for line in (run / "events.jsonl").read_text().splitlines():
    e = json.loads(line)
    if e["kind"] in {"dataset_resolved", "decision", "report", "run_finished"}:
        print(f"cycle {e['cycle']} · {e['stage']:>12} · {e['kind']:<17} {e['payload']}")


In [ ]:
from IPython.display import Video

videos = sorted(run.glob("cycle_*/eval/**/*.mp4"))
Video(videos[0], embed=True, width=480) if videos else print("no eval video found")


## Where to go next

- **LIBERO runs** — `configs/m0_libero.yaml` in the repo: SmolVLA fine-tuning with a
  task-filtered episode selection, policy escalation ladder, and the DexFlyWheel
  self-improvement path (needs MuJoCo/EGL; run `leagents doctor` first).
- **Flow dashboard** — `pip install "leagents[dash]"` then `leagents dash`. In Colab:

  ```python
  import subprocess; subprocess.Popen(["leagents", "dash"])
  from google.colab.output import eval_js
  print(eval_js("google.colab.kernel.proxyPort(8321)"))
  ```

- **LLM proposers** — set `llm: gemini:gemini-2.5-flash` (or `anthropic:*` / `openai:*`)
  in the config; every LLM flow has a deterministic fallback, so `llm: null` always works.
- **Design & research grounding** — [DESIGN.md](https://github.com/ratelcode/LeAgents/blob/main/DESIGN.md);
  the knowledge layer conventions are in section 3.6.
